In [5]:
import pandas as pd
import clickhouse_connect
from datetime import datetime





import os
from dotenv import load_dotenv

load_dotenv()


True

In [6]:

# создаём клиент
# client = clickhouse_connect.get_client(
#     host= os.getenv("CLICKHOUSE_HOST") ,   # замени на свой хост/адрес
#     port= os.getenv("CLICKHOUSE_PORT"),
#     username=  os.getenv("CLICKHOUSE_USER"),
#     password= os.getenv("CLICKHOUSE_PASSWORD")
# )
client = clickhouse_connect.get_client(
    host="84.201.160.255",   # замени на свой хост/адрес
    port=8123,
    username="peter",
    password="1234"
)

In [6]:
# SQL-запрос
query = "SELECT * FROM mailkb.emails ORDER BY id DESC LIMIT 5"

# вернуть сразу DataFrame
df = client.query_df(query)


In [26]:
df = df.iloc[2]['body_text']

In [9]:
## импортируем либы для структурированного ответа

from pydantic import BaseModel, Field
from langchain.agents import create_agent
from typing import List



class MailInfo(BaseModel):
    """Contact information for a person."""
    topic: str = Field(..., description="The name of the mail ")
    email: str = Field(..., description="The email address of the persons of the mail")

class MailChainInfo(BaseModel):
    emails: List[MailInfo] = Field(description='mail chain info')

agent = create_agent(
    model="gpt-5",
    response_format=MailChainInfo  # Auto-selects ProviderStrategy
)

# result = agent.invoke({
#     "messages": [{"role": "user", "content": "Extract contact info from: {}"}]
# })
#
# print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [12]:
df['body_text'][0]

'Это МТЗ?\r\n________________________________\r\n\r\nОт: Ilya M Sergeev <Ilya.Sergeev@ru.ey.com>\r\nОтправлено: воскресенье, 19 августа 2018 г. 14:47\r\nКому: Maxim P Lyutov <Maxim.Lyutov@ru.ey.com>,Chigvintsev Anton <anton.chigvintsev@bearingpoint.ru>,"Kozlov Yuriy(CE)" <ce-yuriy.kozlov@bearingpoint.ru>,Алексей Алмакаев <A.Almakaev@agrohold.ru>,Zhanna L Vakhrusheva <Zhanna.L.Vakhrusheva@ru.ey.com>,Алексей Филимончук <A.Filimonchuk@agrohold.ru>,Андрей Кравченко <A.Kravchenko@agrohold.ru>,Андрей Николаевич Петров <A.N.Petrov@agrohold.ru>,Yuri A Denisov <Yuri.Denisov@ru.ey.com>,Dmitry V Karpov <Dmitry.Karpov@ru.ey.com>,Alexey S Fedoseev <Alexey.Fedoseev@ru.ey.com>,Валерий Бражников <V.Brazhnikov@agrohold.ru>,Vadim Kotenko <V.Kotenko@agrohold.ru>,"Scherbakov Andrey(CE)" <ce-andrey.scherbakov@bearingpoint.ru>,Андрей Владимирович Самойленко <AV.Samoylenko@agrohold.ru>,Вадим Просолупов <V.Prosolupov@agrohold.ru>,Александр Игоревич Александров <a.i.aleksandrov@agrohold.ru>,Иван Лосев <I.Losev

In [10]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": f"""Начни обработку строки и вытащи темы писем и email адреса:

{df['body_text'][0]}   """

        }
    ]
})

report = result["messages"][-1].content
print(report)



{"emails":[{"topic":"RE: Запуск SAP: 6 Филиалов, график координации перехода","email":"Ilya.Sergeev@ru.ey.com"},{"topic":"RE: Запуск SAP: 6 Филиалов, график координации перехода","email":"Maxim.Lyutov@ru.ey.com"},{"topic":"RE: Запуск SAP: 6 Филиалов, график координации перехода","email":"anton.chigvintsev@bearingpoint.ru"},{"topic":"Re: Запуск SAP: 6 Филиалов, график координации перехода","email":"ce-yuriy.kozlov@bearingpoint.ru"},{"topic":"RE: Запуск SAP: 6 Филиалов, график координации перехода","email":"A.Almakaev@agrohold.ru"},{"topic":"RE: Запуск SAP: 6 Филиалов, график координации перехода","email":"A.Filimonchuk@agrohold.ru"},{"topic":"RE: Запуск SAP: 6 Филиалов, график координации перехода","email":"A.Kravchenko@agrohold.ru"},{"topic":"RE: Запуск SAP: 6 Филиалов, график координации перехода","email":"Yuri.Denisov@ru.ey.com"},{"topic":"RE: Запуск SAP: 6 Филиалов, график координации перехода","email":"Alexey.Fedoseev@ru.ey.com"},{"topic":"Re: Запуск SAP: 6 Филиалов, график координ

In [12]:
Structured output


ключевые поинты
emailы все

SyntaxError: invalid syntax (734425773.py, line 1)

### Агент по очистке письма в диалогах

In [62]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from clickhouse_connect import get_client
import hashlib
import time

In [50]:
# --- structured output schema ---

class CleanBody(BaseModel):
    """Cleaned body of an email."""

    body_clean: str = Field(
        ...,
        description="Top-level email text without quoted history, headers, or signatures"
    )

# --- prompt for the LLM ---

PROMPT = """
You clean email bodies.

Task:
Extract ONLY the new message written by the sender.

Remove:
- quoted history
- repeated headers (From, Sent, Subject)
- signatures
- reply separators

Keep only the actual message content.

Return JSON matching the schema.
"""

# --- create LLM agent ---

agent = create_agent(
    model="deepseek-chat",
    response_format=CleanBody
)


In [57]:
# ---------- helper ----------

def body_md5(text):

    text = text.strip()

    return hashlib.md5(text.encode()).hexdigest()


def clean_email_body(text: str):

    result = agent.invoke({
        "messages": [
            {"role": "system", "content": PROMPT},
            {"role": "user", "content": text}
        ]
    })

    return result["structured_response"].body_clean


In [58]:
# ---------- загрузим кэш md5 в память ----------

print("Loading cache...")

cached_md5 = set(
    r[0] for r in client.query("""
        SELECT raw_md5
        FROM mailkb.llm_body_clean_cache
    """).result_rows
)

print("Cached bodies:", len(cached_md5))

Loading cache...
Cached bodies: 48


In [52]:
# body = df["body_text"][0]
body = df
cleaned = clean_email_body(body)




In [59]:
cleaned

'Добрый день!\n\nХотел у вас уточнить статус по тм014, когда можно продолжать тестировать с исправлениями?\n\nС уважением, Острик Петр'

In [67]:
# ---------- batch параметры ----------

BATCH = 2

In [68]:
# ---------- основной цикл ----------

while True:

    rows = client.query("""
        SELECT
            id,
            body_text
        FROM mailkb.emails
        WHERE body_text IS NOT NULL
          AND body_text != ''
        LIMIT %(limit)s
    """, {"limit": BATCH}).result_rows

    if not rows:
        print("No rows found")
        break

    insert_batch = []

    for email_id, body in rows:

        md5 = body_md5(body)

        # если уже обработано — пропускаем
        if md5 in cached_md5:
            continue

        start = time.time()

        try:

            cleaned = clean_email_body(body)

            latency = int((time.time() - start) * 1000)

            insert_batch.append({
                "raw_md5": md5,
                "parser_version": "v1",
                "model_name": "deepseek-chat",
                "status": "success",
                "body_clean": cleaned,
                "error": "",
                "tokens_in": 0,
                "tokens_out": 0,
                "latency_ms": latency
            })

            cached_md5.add(md5)

        except Exception as e:

            insert_batch.append({
                "raw_md5": md5,
                "parser_version": "v1",
                "model_name": "deepseek-chat",
                "status": "failed",
                "body_clean": "",
                "error": str(e),
                "tokens_in": 0,
                "tokens_out": 0,
                "latency_ms": 0
            })

        print("Processing:", email_id)

if insert_batch:

    data = [
        [
            row["raw_md5"],
            row["parser_version"],
            row["model_name"],
            row["status"],
            row["body_clean"],
            row["error"],
            row["tokens_in"],
            row["tokens_out"],
            row["latency_ms"]
        ]
        for row in insert_batch
    ]

    client.insert(
        "mailkb.llm_body_clean_cache",
        data,
        column_names=[
            "raw_md5",
            "parser_version",
            "model_name",
            "status",
            "body_clean",
            "error",
            "tokens_in",
            "tokens_out",
            "latency_ms"
        ]
    )

    print("Inserted:", len(data))


KeyboardInterrupt: 

### batch model agent
чистим батчами


In [69]:
import hashlib
import time
from clickhouse_connect import get_client
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from typing import List


# ---------- LLM schema (batch) ----------

class CleanItem(BaseModel):
    raw_md5: str = Field(description="MD5 of the original raw body (as provided in the input batch)")
    body_clean: str = Field(description="Cleaned top-level email text without quoted history, headers or signatures")

class CleanBatch(BaseModel):
    items: List[CleanItem]

In [71]:
# ---------- LLM agent ----------

agent = create_agent(
    model="deepseek-chat",
    response_format=CleanBatch
)

PROMPT = """
You clean email bodies.

Task:
For each email body, extract ONLY the new message written by the sender.

Remove:
- quoted history (previous replies/forwards)
- repeated headers like From/Sent/Subject
- signatures, phone blocks, company footers

Return JSON matching the schema.

Important:
- DO NOT invent content.
- Return one item per input email with the same raw_md5.
"""


# ---------- ClickHouse ----------

client = clickhouse_connect.get_client(
    host="84.201.160.255",   # замени на свой хост/адрес
    port=8123,
    username="peter",
    password="1234"
)

# ---------- helpers ----------

def body_md5(text: str) -> str:
    text = (text or "").strip()
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def clean_email_bodies_batch(md5_and_text: List[tuple[str, str]]) -> dict[str, str]:
    """
    md5_and_text: list of (raw_md5, body_text)
    returns: dict raw_md5 -> body_clean
    """

    # Собираем вход в понятном для модели виде
    # (важно: включаем raw_md5, чтобы можно было сопоставить ответы)
    blocks = []
    for md5, body in md5_and_text:
        blocks.append(f"raw_md5: {md5}\nbody:\n{body}")

    user_content = "\n\n---\n\n".join(blocks)

    result = agent.invoke({
        "messages": [
            {"role": "system", "content": PROMPT},
            {"role": "user", "content": user_content}
        ]
    })

    batch_obj: CleanBatch = result["structured_response"]

    # Превращаем в dict для удобства
    out = {}
    for item in batch_obj.items:
        out[item.raw_md5] = item.body_clean

    return out


In [73]:
# ---------- load cache md5 to memory ----------

print("Loading cache...")
cached_md5 = set(r[0] for r in client.query("""
    SELECT raw_md5 FROM mailkb.llm_body_clean_cache
""").result_rows)
print("Cached:", len(cached_md5))

Loading cache...
Cached: 48


In [80]:
# ---------- parameters ----------

FETCH_BATCH = 30      # сколько строк читать из emails за раз
LLM_BATCH = 5         # сколько писем отправлять в один LLM запрос (начни с 5-10)
PARSER_VERSION = "v1"
MODEL_NAME = "deepseek-chat"


In [81]:
# ---------- main loop ----------

while True:
    # Для простоты: берём "какие-то" письма (как в твоём шаге 5),
    # но обязательно добавь ORDER BY rand() на тесте, чтобы не крутиться на одном и том же.
    rows = client.query("""
        SELECT id, body_text
        FROM mailkb.emails
        WHERE body_text IS NOT NULL AND body_text != ''
        ORDER BY rand()
        LIMIT %(limit)s
    """, {"limit": FETCH_BATCH}).result_rows

    if not rows:
        print("No rows found, stop.")
        break

    # Отфильтруем уже обработанные по md5
    to_process = []
    for email_id, body in rows:
        md5 = body_md5(body)
        if md5 in cached_md5:
            continue
        to_process.append((md5, body))

    if not to_process:
        print("All fetched rows already cached; fetching next...")
        continue

    # Разбиваем на пачки для LLM
    for i in range(0, len(to_process), LLM_BATCH):
        chunk = to_process[i:i+LLM_BATCH]

        start = time.time()

        try:
            cleaned_map = clean_email_bodies_batch(chunk)
            latency_ms = int((time.time() - start) * 1000)

            # Подготовим данные для вставки в CH: list of lists + column_names
            data = []
            for raw_md5, _body in chunk:
                body_clean = cleaned_map.get(raw_md5, "")
                status = "success" if body_clean.strip() else "failed"
                error = "" if status == "success" else "empty_clean_result"

                data.append([
                    raw_md5,
                    PARSER_VERSION,
                    MODEL_NAME,
                    status,
                    body_clean,
                    error,
                    0,   # tokens_in (пока 0, потом можно заполнить, если у клиента есть usage)
                    0,   # tokens_out
                    latency_ms
                ])

                cached_md5.add(raw_md5)

            client.insert(
                "mailkb.llm_body_clean_cache",
                data,
                column_names=[
                    "raw_md5",
                    "parser_version",
                    "model_name",
                    "status",
                    "body_clean",
                    "error",
                    "tokens_in",
                    "tokens_out",
                    "latency_ms"
                ]
            )

            print(f"Inserted batch: {len(data)} (latency_ms={latency_ms})")

        except Exception as e:
            # Если пачка упала — не теряем всё: можно записать failed для всех md5 в chunk
            err = str(e)
            data = []
            for raw_md5, _body in chunk:
                data.append([
                    raw_md5,
                    PARSER_VERSION,
                    MODEL_NAME,
                    "failed",
                    "",
                    err,
                    0,
                    0,
                    0
                ])
                cached_md5.add(raw_md5)

            client.insert(
                "mailkb.llm_body_clean_cache",
                data,
                column_names=[
                    "raw_md5",
                    "parser_version",
                    "model_name",
                    "status",
                    "body_clean",
                    "error",
                    "tokens_in",
                    "tokens_out",
                    "latency_ms"
                ]
            )

            print(f"Batch failed, inserted failed rows: {len(data)}; error={err}")

Inserted batch: 5 (latency_ms=50381)
Inserted batch: 5 (latency_ms=25194)
Inserted batch: 5 (latency_ms=31707)
Inserted batch: 5 (latency_ms=21130)
Inserted batch: 5 (latency_ms=34198)
Inserted batch: 5 (latency_ms=17904)
Inserted batch: 5 (latency_ms=53076)
Inserted batch: 5 (latency_ms=27121)
Inserted batch: 5 (latency_ms=24617)
Inserted batch: 5 (latency_ms=22815)
Inserted batch: 5 (latency_ms=23213)
Inserted batch: 3 (latency_ms=28559)
Inserted batch: 5 (latency_ms=42284)
Inserted batch: 5 (latency_ms=49179)
Inserted batch: 5 (latency_ms=18030)
Inserted batch: 5 (latency_ms=18800)
Inserted batch: 5 (latency_ms=36437)
Inserted batch: 5 (latency_ms=40306)
Inserted batch: 5 (latency_ms=52406)
Inserted batch: 5 (latency_ms=16687)
Inserted batch: 5 (latency_ms=60343)
Inserted batch: 5 (latency_ms=20106)
Inserted batch: 5 (latency_ms=22168)
Inserted batch: 2 (latency_ms=27935)
Batch failed, inserted failed rows: 5; error=Error code: 400 - {'error': {'message': "An assistant message with 

KeyboardInterrupt: 

### вложенный класс


In [83]:
import pandas as pd
import clickhouse_connect
from datetime import datetime
import os
from dotenv import load_dotenv

load_dotenv()


True

In [84]:

# создаём клиент
# client = clickhouse_connect.get_client(
#     host= os.getenv("CLICKHOUSE_HOST") ,   # замени на свой хост/адрес
#     port= os.getenv("CLICKHOUSE_PORT"),
#     username=  os.getenv("CLICKHOUSE_USER"),
#     password= os.getenv("CLICKHOUSE_PASSWORD")
# )
client = clickhouse_connect.get_client(
    host="84.201.160.255",   # замени на свой хост/адрес
    port=8123,
    username="peter",
    password="1234"
)

In [90]:
# SQL-запрос
query = "SELECT * FROM mailkb.emails ORDER BY id DESC LIMIT 5"

# вернуть сразу DataFrame
df = client.query_df(query)


In [91]:
# df = df.iloc[2]['body_text']

In [92]:
df

,id,message_id,subject,from_addr,to_addr,cc_addr,bcc_addr,sent_at_utc,sent_at_raw,folder,body_text,body_html,subject_norm,thread_id
0,jxQhgOJ7J0GmAtuWNDdmMQAAANUzoWuqRNoH9-5MgJqWgq...,jxQhgOJ7J0GmAtuWNDdmMQAAANUzoWuqRNoH9-5MgJqWgq...,"RE: Запуск SAP: 6 Филиалов, график координаци...",[Maxim.Lyutov@ru.ey.com],"[Ilya.Sergeev@ru.ey.com, anton.chigvintsev@bea...",[],[],2018-08-19 11:48:48,"Sun, 19 Aug 2018 11:48:48 +0000",unknown,Это МТЗ?\r\n________________________________\r...,"<html><head>\r\n<meta http-equiv=""Content-Type...","Запуск sap: 6 Филиалов, график координации пе...",d314be6c1694dcc877bf114422956bba
1,jxQhgOJ7J0GmAtuWNDdmMQAAANUzoWuqRNoH9-5MgJqWgq...,jxQhgOJ7J0GmAtuWNDdmMQAAANUzoWuqRNoH9-5MgJqWgq...,"RE: Запуск SAP: 6 Филиалов, график координаци...",[Maxim.Lyutov@ru.ey.com],"[anton.chigvintsev@bearingpoint.ru, ce-rustam....",[],[],2018-08-19 12:54:33,"Sun, 19 Aug 2018 12:54:33 +0000",unknown,"Илья, Антн,\r\nдобавьте статус гашения Всд в д...","<html><head>\r\n<meta http-equiv=""Content-Type...","Запуск sap: 6 Филиалов, график координации пе...",d314be6c1694dcc877bf114422956bba
2,jxQhgOJ7J0GmAtuWNDdmMQAAANUzoWuqRNoH9-5MgJqWgq...,jxQhgOJ7J0GmAtuWNDdmMQAAANUzoWuqRNoH9-5MgJqWgq...,"RE: Запуск SAP: 6 Филиалов, график координаци...",[Maxim.Lyutov@ru.ey.com],"[anton.chigvintsev@bearingpoint.ru, ce-yuriy.k...",[],[],2018-08-19 11:38:13,"Sun, 19 Aug 2018 11:38:13 +0000",unknown,"Илья Сергеев, проверь пожалуйста наличие всд в...","<html><head>\r\n<meta http-equiv=""Content-Type...","Запуск sap: 6 Филиалов, график координации пе...",d314be6c1694dcc877bf114422956bba
3,archive.pst::3027620,,,[],[],[],[],1970-01-01 00:00:00,,unknown,szeligowska_ea@segezha-group.com 16:27: \r\nПё...,"<html xmlns=""http://schemas.microsoft.com/2008...",no_subject,7c4b5aaa31326e5523268cd24c3466cf
4,archive.pst::3024932,,,[],[],[],[],1970-01-01 00:00:00,,unknown,Пропущенный звонок конференц-связи от пользова...,"<html xmlns=""http://schemas.microsoft.com/2008...",no_subject,7c4b5aaa31326e5523268cd24c3466cf


#### вариант 1

In [99]:
## импортируем либы для структурированного ответа

from pydantic import BaseModel, Field
from langchain.agents import create_agent
from typing import List



class MailInfo(BaseModel):
    """Contact information for a person."""
    topic: str = Field(..., description="The name of the mail ")
    email_body: str = Field(..., description="The email clean body text")
    date: str = Field(..., description="The date and time the message was sent. Format: YYYY-MM-DD")
    mail_query_number: int = Field(..., description="The mail query in topic number")
    key_words

class MailChainInfo(BaseModel):
    emails: List[MailInfo] = Field(description='mail chain info')
    #темы

agent = create_agent(
    model="",
    response_format=MailChainInfo  # Auto-selects ProviderStrategy
)

# result = agent.invoke({
#     "messages": [{"role": "user", "content": "Extract contact info from: {}"}]
# })
#
# print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [97]:
df['body_text'][0]

'Это МТЗ?\r\n________________________________\r\n\r\nОт: Ilya M Sergeev <Ilya.Sergeev@ru.ey.com>\r\nОтправлено: воскресенье, 19 августа 2018 г. 14:47\r\nКому: Maxim P Lyutov <Maxim.Lyutov@ru.ey.com>,Chigvintsev Anton <anton.chigvintsev@bearingpoint.ru>,"Kozlov Yuriy(CE)" <ce-yuriy.kozlov@bearingpoint.ru>,Алексей Алмакаев <A.Almakaev@agrohold.ru>,Zhanna L Vakhrusheva <Zhanna.L.Vakhrusheva@ru.ey.com>,Алексей Филимончук <A.Filimonchuk@agrohold.ru>,Андрей Кравченко <A.Kravchenko@agrohold.ru>,Андрей Николаевич Петров <A.N.Petrov@agrohold.ru>,Yuri A Denisov <Yuri.Denisov@ru.ey.com>,Dmitry V Karpov <Dmitry.Karpov@ru.ey.com>,Alexey S Fedoseev <Alexey.Fedoseev@ru.ey.com>,Валерий Бражников <V.Brazhnikov@agrohold.ru>,Vadim Kotenko <V.Kotenko@agrohold.ru>,"Scherbakov Andrey(CE)" <ce-andrey.scherbakov@bearingpoint.ru>,Андрей Владимирович Самойленко <AV.Samoylenko@agrohold.ru>,Вадим Просолупов <V.Prosolupov@agrohold.ru>,Александр Игоревич Александров <a.i.aleksandrov@agrohold.ru>,Иван Лосев <I.Losev

In [98]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": f"""Начни обработку строки и вытащи темы писем и email адреса:

{df['body_text'][0]}   """

        }
    ]
})

report = result["messages"][-1].content
print(report)



StructuredOutputValidationError: Failed to parse structured output for tool 'MailChainInfo': Failed to parse data to MailChainInfo: 3 validation errors for MailChainInfo
emails.0.date
  Input should be a valid datetime or date, input is too short [type=datetime_from_date_parsing, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/datetime_from_date_parsing
emails.1.date
  Input should be a valid datetime or date, input is too short [type=datetime_from_date_parsing, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/datetime_from_date_parsing
emails.2.date
  Input should be a valid datetime or date, input is too short [type=datetime_from_date_parsing, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/datetime_from_date_parsing.

#### вариант 2 (12.03.26 -работает)

In [4]:

from pydantic import BaseModel, Field
from typing import List
from langchain.agents import create_agent


class MailInfo(BaseModel):
    topic: str = Field(description="Topic of the email")
    email_body: str = Field(description="Clean email body")
    date: str = Field(description="Date of the email YYYY-MM-DD")
    mail_query_number: int = Field(description="Order number of the email in the chain")
    key_words: List[str] = Field(description="Important keywords")


class MailChainInfo(BaseModel):
    emails: List[MailInfo] = Field(description="Mail chain info")

In [5]:

agent = create_agent(
    model="deepseek-chat",
    tools=[],  # явно без инструментов
    response_format=MailChainInfo
)

In [9]:
query = """
SELECT
    id,
    sent_at_utc,
    body_text
FROM mailkb.emails_unique

LEFT ANTI JOIN mailkb.mail_parsed
ON emails_unique.id = mail_parsed.email_id

WHERE
    body_text IS NOT NULL
    AND length(body_text) > 30

ORDER BY sent_at_utc DESC
LIMIT 10
"""

df = client.query_df(query)

In [10]:
df

,id,sent_at_utc,body_text
0,8977d82a-f23a-47d7-9da9-537cd742d704,2021-02-28 20:22:34+00:00,Добрый день! Необходимо проставить оценку по ц...
1,47e3ef2c-9329-4ecb-9c46-925aedcebc9b,2021-02-28 20:02:00+00:00,—-—-—-— \nReply above this line. \n\nШэлиговск...
2,71c6e416-4956-42bc-ae42-09250ef28af0,2021-02-28 20:01:00+00:00,<https://www.gravatar.com/avatar/bc1100ceff6b...
3,8a07aa9e-303f-4b17-b729-0abb724219f7,2021-02-28 16:51:43+00:00,"Коллеги, добрый день\n\n \n\nПо данному вопрос..."
4,c73247b7-5efd-4b83-8082-74be467a32fa,2021-02-26 14:57:00+00:00,<https://www.gravatar.com/avatar/e6ebe0ca1152...
5,1ba5fb9f-edd8-44d3-bc3f-585c4987fff0,2021-02-26 14:41:01+00:00,—-—-—-— \nReply above this line. \n\nОстрик Пё...
6,a2d84c74-dcd7-4ac5-a942-2158f4be3b77,2021-02-26 14:12:00+00:00,—-—-—-— \nReply above this line. \n\nШарманов ...
7,c85daa0b-5eca-41e3-84ff-021be5a91798,2021-02-26 13:59:00+00:00,—-—-—-— \nReply above this line. \n\nСафина Ай...
8,224247aa-f10f-44fe-89f7-ce1d9e054da3,2021-02-26 13:58:00+00:00,<https://www.gravatar.com/avatar/1701f769b237...
9,ca0eb357-e8c3-4b76-b3bc-9696d9e23fbd,2021-02-26 13:41:02+00:00,<https://www.gravatar.com/avatar/572b97488d33...


In [11]:
results = []

for i in range(len(df)):
    body = df["body_text"][i]

    result = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": f"""
Начни обработку письма.

Извлеки структуру переписки.

EMAIL BODY:
{body}
"""
            }
        ]
    })

    structured = result["structured_response"]
    parsed_json = structured.model_dump_json()

    results.append({
        "email_id": df["id"][i],
        "parsed_json": parsed_json
    })


In [12]:
results

[{'email_id': '8977d82a-f23a-47d7-9da9-537cd742d704',
  'parsed_json': '{"emails":[{"topic":"Оценка стажера Генисаретский Сергей Сергеевич","email_body":"Добрый день! Необходимо проставить оценку по целям за 6 месяцев работы стажера Генисаретский Сергей Сергеевич и подтвердить его готовность к стажерской панели.\\n\\nПерейти в 1С:Документооборот <https://do.bearingpoint.ru/do/ru_RU//#e1cib/data/Документ.ОценкаСтажеров?ref=812b000c291ff2d711eaf083ed3b81f5> \\n\\nС уважением,\\n\\nОтдел по управлению персоналом","date":"2024-01-01","mail_query_number":1,"key_words":["оценка","стажер","Генисаретский Сергей Сергеевич","6 месяцев","цели","стажерская панель","1С:Документооборот","управление персоналом"]}]}'},
 {'email_id': '47e3ef2c-9329-4ecb-9c46-925aedcebc9b',
  'parsed_json': '{"emails":[{"topic":"Настройка профиля SAPSP-18265","email_body":"Коллеги, профиль настроили?\\n\\nView request <http://192.168.210.55:8080/servicedesk/customer/portal/2/SAPSP-18265?sda_source=notification-email>  ·

In [13]:

rows = [
    [r["email_id"], r["parsed_json"]]
    for r in results
]

client.insert(
    "mailkb.mail_parsed",
    rows,
    column_names=["email_id", "parsed_json"]
)

#### вариант 3 с параллельной обработкой

In [3]:
from pydantic import BaseModel, Field
from typing import List
from langchain.agents import create_agent
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd


class MailInfo(BaseModel):
    topic: str = Field(description="Topic of the email")
    email_body: str = Field(description="Clean email body")
    date: str = Field(description="Date of the email YYYY-MM-DD")
    mail_query_number: int = Field(description="Order number of the email in the chain")
    key_words: List[str] = Field(description="Important keywords")


class ParsedEmailResult(BaseModel):
    email_id: str = Field(description="Original email id from the input batch")
    emails: List[MailInfo] = Field(description="Parsed content for this email")


class ParsedEmailBatch(BaseModel):
    items: List[ParsedEmailResult] = Field(description="Parsed results for all emails in the batch")


agent = create_agent(
    model="deepseek-chat",
    tools=[],
    response_format=ParsedEmailBatch
)


query = """
SELECT
    id,
    sent_at_utc,
    body_text
FROM mailkb.emails_unique

LEFT ANTI JOIN mailkb.mail_parsed
ON emails_unique.id = mail_parsed.email_id

WHERE
    body_text IS NOT NULL
    AND length(body_text) > 30

ORDER BY sent_at_utc DESC
LIMIT 30
"""


In [7]:

df = client.query_df(query)


def chunk_list(rows, batch_size):
    for i in range(0, len(rows), batch_size):
        yield rows[i:i + batch_size]


def build_batch_prompt(batch_rows):
    parts = []
    for idx, row in enumerate(batch_rows, start=1):
        body = (row["body_text"] or "")[:15000]
        parts.append(
            f"""
EMAIL #{idx}
email_id: {row["id"]}

EMAIL BODY:
{body}
""".strip()
        )

    return """
Начни обработку нескольких писем.

Для КАЖДОГО письма:
- извлеки структуру переписки,
- верни отдельный результат,
- обязательно сохрани правильный email_id,
- не смешивай письма между собой.

Верни результат для всех писем из входного батча.

{}
""".format("\n\n" + ("\n\n" + "=" * 80 + "\n\n").join(parts))


def process_batch(batch_rows):
    try:
        prompt = build_batch_prompt(batch_rows)

        result = agent.invoke({
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        })

        structured = result["structured_response"]

        parsed_rows = []
        returned_ids = set()

        for item in structured.items:
            returned_ids.add(item.email_id)
            parsed_rows.append({
                "email_id": item.email_id,
                "parsed_json": item.model_dump_json()
            })

        # если модель не вернула часть писем — помечаем ошибкой
        requested_ids = {row["id"] for row in batch_rows}
        missing_ids = requested_ids - returned_ids

        error_rows = [
            {"email_id": mid, "error": "missing_in_model_response"}
            for mid in missing_ids
        ]

        return parsed_rows, error_rows

    except Exception as e:
        return [], [
            {"email_id": row["id"], "error": str(e)}
            for row in batch_rows
        ]



In [8]:

BATCH_SIZE = 3
MAX_WORKERS = 6

rows = df.to_dict("records")
batches = list(chunk_list(rows, BATCH_SIZE))

all_success = []
all_errors = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_batch, batch) for batch in batches]

    for future in as_completed(futures):
        success_rows, error_rows = future.result()
        all_success.extend(success_rows)
        all_errors.extend(error_rows)

print("success:", len(all_success))
print("errors:", len(all_errors))


success: 9
errors: 21


In [9]:

# успешные результаты — в ClickHouse
if all_success:
    insert_rows = [[r["email_id"], r["parsed_json"]] for r in all_success]
    client.insert(
        "mailkb.mail_parsed",
        insert_rows,
        column_names=["email_id", "parsed_json"]
    )

errors_df = pd.DataFrame(all_errors)

In [10]:
errors_df

,email_id,error
0,10532f64-e0f3-4e98-bf02-0297f70e4616,"Error code: 400 - {'error': {'message': ""An as..."
1,79cd1c4f-39b4-4d7e-9083-a8a2c81ffc88,"Error code: 400 - {'error': {'message': ""An as..."
2,1a94651d-9380-4794-9330-7eca2218f93f,"Error code: 400 - {'error': {'message': ""An as..."
3,036e6e6e-2313-480d-9915-bffa5e0d1c00,"Error code: 400 - {'error': {'message': ""An as..."
4,8c095f6d-fd68-41f7-920a-c0bcc8959d6f,"Error code: 400 - {'error': {'message': ""An as..."
5,fc461be4-400f-4138-b06d-62dbb1a36130,"Error code: 400 - {'error': {'message': ""An as..."
6,1b4a4ed5-d8d1-40b4-98dc-e8088bb1feab,"Error code: 400 - {'error': {'message': ""An as..."
7,d9f680a9-3ed4-4805-8488-23849334c7bc,"Error code: 400 - {'error': {'message': ""An as..."
8,bc9899a3-98ed-4044-aa38-f43604c9cf58,"Error code: 400 - {'error': {'message': ""An as..."
9,bd2f2c30-82d9-4f7e-a612-dde8407a0459,"Error code: 400 - {'error': {'message': ""An as..."


#### вариант 4 параллельная обработка без агента

In [11]:
from pydantic import BaseModel, Field
from typing import List


class MailInfo(BaseModel):
    topic: str
    email_body: str
    date: str
    mail_query_number: int
    key_words: List[str]


class ParsedEmailResult(BaseModel):
    email_id: str
    emails: List[MailInfo]


class ParsedEmailBatch(BaseModel):
    items: List[ParsedEmailResult]

In [12]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="deepseek-chat",
    temperature=0
)

structured_llm = llm.with_structured_output(ParsedEmailBatch)

In [13]:
def build_batch_prompt(batch_rows):
    parts = []

    for idx, row in enumerate(batch_rows, start=1):
        body = (row["body_text"] or "")[:15000]

        parts.append(
            f"""
EMAIL #{idx}
email_id: {row["id"]}

EMAIL BODY:
{body}
""".strip()
        )

    return f"""
Начни обработку нескольких писем.

Для КАЖДОГО письма:
- извлеки структуру переписки
- верни результат отдельно
- ОБЯЗАТЕЛЬНО укажи email_id
- не смешивай письма между собой

Верни результат в формате ParsedEmailBatch.

{chr(10).join(parts)}
"""

In [14]:
import time


def call_with_retry(prompt, max_retries=3):

    for attempt in range(max_retries):
        try:
            return structured_llm.invoke(prompt)

        except Exception as e:
            print(f"Retry {attempt+1} due to error:", e)
            time.sleep(1.5 * (attempt + 1))

    raise Exception("LLM failed after retries")

In [15]:
def process_batch(batch_rows):

    prompt = build_batch_prompt(batch_rows)

    try:
        structured = call_with_retry(prompt)

        parsed_rows = []
        returned_ids = set()

        for item in structured.items:
            returned_ids.add(item.email_id)

            parsed_rows.append({
                "email_id": item.email_id,
                "parsed_json": item.model_dump_json()
            })

        requested_ids = {row["id"] for row in batch_rows}
        missing_ids = requested_ids - returned_ids

        error_rows = [
            {"email_id": mid, "error": "missing_in_response"}
            for mid in missing_ids
        ]

        return parsed_rows, error_rows

    except Exception as e:
        return [], [
            {"email_id": row["id"], "error": str(e)}
            for row in batch_rows
        ]

In [16]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def chunk_list(rows, batch_size):
    for i in range(0, len(rows), batch_size):
        yield rows[i:i + batch_size]


query = """
SELECT
    id,
    sent_at_utc,
    body_text
FROM mailkb.emails_unique

LEFT ANTI JOIN mailkb.mail_parsed
ON emails_unique.id = mail_parsed.email_id

WHERE
    body_text IS NOT NULL
    AND length(body_text) > 30

ORDER BY sent_at_utc DESC
LIMIT 50
"""

df = client.query_df(query)

rows = df.to_dict("records")

BATCH_SIZE = 3
MAX_WORKERS = 6

batches = list(chunk_list(rows, BATCH_SIZE))

all_success = []
all_errors = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = [executor.submit(process_batch, b) for b in batches]

    for future in as_completed(futures):
        success_rows, error_rows = future.result()

        all_success.extend(success_rows)
        all_errors.extend(error_rows)

print("SUCCESS:", len(all_success))
print("ERRORS:", len(all_errors))

Retry 1 due to error: Function ParsedEmailBatch arguments:

{"items": [{"email_id": "10532f64-e0f3-4e98-bf02-0297f70e4616", "emails": [{"topic": "ошибки ТМ", "email_body": "Посмотрела все нечаянно )\n\nПо транспореону написала письмо Шэлиговска.\n\nПо остальным отменила, а ошибки по iso код обработал Сергей Г.\n\n \n\nFrom: Ostrik Petr \nSent: Wednesday, February 17, 2021 2:31 PM\nTo: Safina Aygul <aygul.safina@bearingpoint.ru>\nSubject: FW: ошибки ТМ\n\n \n\nПосмотри плз Транспореон\n\n \n\nFrom: Pinkhasov Dmitriy(CE) \nSent: Thursday, February 11, 2021 11:30 PM\nTo: Ostrik Petr <petr.ostrik@bearingpoint.ru <mailto:petr.ostrik@bearingpoint.ru> >\nSubject: RE: ошибки ТМ\n\n \n\nПривет\n\n \n\nПосмотри пжл ошибки в STP\n\nПо TRANSPOREON\n\n36D4EBF361FD11EB80E300000C24F6BA\n\nC7A7C644647611EB847000000C24F6BA\n\n45714745651D11EB914F00000C24F6BB\n\nBA8BE2B0655F11EBB37C00000C24F6BA\n\n70531C27663211EB8A1F00000C24F6BA\n\n32258A5366B411EB9F4A00000C24F6BB\n\n35CC95BD66B411EBC5E500000C24F6BB\n\

KeyboardInterrupt: 

In [ ]:
if all_success:
    insert_rows = [
        [r["email_id"], r["parsed_json"]]
        for r in all_success
    ]

    client.insert(
        "mailkb.mail_parsed",
        insert_rows,
        column_names=["email_id", "parsed_json"]
    )

In [ ]:
import pandas as pd

errors_df = pd.DataFrame(all_errors)
# errors_df.to_csv("mail_errors.csv", index=False)